In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingWarmRestarts, SequentialLR
from torch.utils.data import Dataset, DataLoader
import math
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import time as pytime
import argparse
import os
from scipy.stats import pearsonr
from pathlib import Path

In [2]:
adata = sc.read('./Data/FinalData/adataAfterClean.h5ad')
gptEmbed_df = pd.read_csv('./Data/FinalData/gptEmbed_Jul9_final.csv',index_col=0)
MFP_df = pd.read_csv("./Data/FinalData/compounds_512MFP_wholeDat_fixed.csv",index_col=0)
drug_targets = pd.read_csv("./Data/FinalData/compounds_target_multihot_full.csv", index_col=0)
sc.pp.normalize_total(adata)

In [3]:
adata

AnnData object with n_obs × n_vars = 883077 × 978
    obs: 'cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'plate', 'random_split1', 'cell_split1', 'drug_split1', 'random_split2', 'cell_split2', 'drug_split2', 'random_split3', 'cell_split3', 'drug_split3', 'random_split4', 'cell_split4', 'drug_split4', 'random_split5', 'cell_split5', 'drug_split5'

In [4]:
adata.obs['cell_split2'].value_counts()

cell_split2
train    730869
valid     80038
          46428
test      25742
Name: count, dtype: int64

In [ ]:
'''
use baseline gene expression
'''

In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from typing import Tuple, Optional

def compute_overall_baseline_metrics(
    adata,
    split_col: str = "random_split1",
    batch_size: int = 20000,
    dtype: str = "float32",
) -> Tuple[float, float, float, dict]:
    """
    Paired-control baseline evaluation.
    For each TEST row, prediction = its paired control row (via `obs['paired_control_index']`).
    Returns:
      - overall_mse: mean of squared error over ALL genes and ALL test rows
      - mean_r2:     average (nanmean) of per-sample R² across genes
      - mean_pcc:    average (nanmean) of per-sample Pearson correlation across genes
      - stats:       dict with counts used (n_test_used, n_r2_used, n_pcc_used)

    Notes
    -----
    - Works whether `paired_control_index` stores control *labels* (obs_names) or *row positions* (ints).
    - Per-sample R²_i uses: R²_i = 1 - SSE_i/SST_i with SST_i computed around the sample’s observed gene mean.
      If SST_i == 0, R²_i is set to NaN.
    - Per-sample PCC_i is Pearson correlation across the 978 genes. If either vector has zero variance, PCC_i = NaN.
    - R² and PCC are averaged across samples with `np.nanmean`.
    """

    if split_col not in adata.obs.columns:
        raise KeyError(f"'{split_col}' not found in adata.obs")
    if "paired_control_index" not in adata.obs.columns:
        raise KeyError("'paired_control_index' not found in adata.obs")

    # Identify test rows
    test_mask = (adata.obs[split_col].astype(str).str.lower() == "test")
    if not test_mask.any():
        raise ValueError(f"No rows labeled 'test' under column '{split_col}'.")

    # Resolve paired controls: prefer label mapping; fall back to numeric
    pci_series = adata.obs.loc[test_mask, "paired_control_index"]
    obs_names = adata.obs_names.astype(str)
    pci_as_str = pci_series.astype(str)
    ctrl_idx_from_labels = pd.Index(obs_names).get_indexer(pci_as_str)
    label_ok = ctrl_idx_from_labels >= 0

    numeric_try = pd.to_numeric(pci_series[~label_ok], errors="coerce")
    numeric_ok = numeric_try.notna() & (numeric_try.astype(float) >= 0) & (numeric_try.astype(float) < adata.n_obs)

    test_idx_all = np.where(test_mask.values)[0]
    keep_mask = np.zeros_like(test_idx_all, dtype=bool)
    ctrl_idx_all = np.empty_like(test_idx_all)

    if label_ok.any():
        pos = np.where(label_ok)[0]
        keep_mask[pos] = True
        ctrl_idx_all[pos] = ctrl_idx_from_labels[label_ok]
    if numeric_ok.any():
        pos_num = np.where(~label_ok)[0][numeric_ok.values]
        keep_mask[pos_num] = True
        ctrl_idx_all[pos_num] = numeric_try.loc[numeric_try.index[numeric_ok]].astype(int).values

    test_idx_all = test_idx_all[keep_mask]
    ctrl_idx_all = ctrl_idx_all[keep_mask]

    if test_idx_all.size == 0:
        raise ValueError("After resolving `paired_control_index` as labels/indices, no valid test pairs remain.")

    X = adata.X
    n_genes = adata.n_vars

    def _rows_to_array(row_idx: np.ndarray):
        if issparse(X):
            return X[row_idx].toarray().astype(dtype, copy=False)
        return np.asarray(X[row_idx], dtype=dtype)

    total_sse = 0.0            # for overall MSE
    r2_list = []               # collect per-sample R²
    pcc_list = []              # collect per-sample PCC

    # Stream in batches
    for start in range(0, test_idx_all.size, batch_size):
        end = min(start + batch_size, test_idx_all.size)
        ti = test_idx_all[start:end]
        ci = ctrl_idx_all[start:end]

        Xt = _rows_to_array(ti)  # (b, n_genes) observed perturbed
        Xc = _rows_to_array(ci)  # (b, n_genes) baseline prediction (paired control)

        # ----- Overall MSE accumulation -----
        diff = Xt - Xc
        total_sse += np.square(diff, out=diff).sum(dtype="float64")

        # ----- Per-sample R² across genes -----
        # SSE_i
        sse_i = np.square(Xt - Xc).sum(axis=1, dtype="float64")
        # SST_i around each sample's observed gene mean
        y_mean = Xt.mean(axis=1, dtype="float64", keepdims=True)        # (b,1)
        sst_i = np.square(Xt - y_mean).sum(axis=1, dtype="float64")     # (b,)
        # Avoid divide-by-zero: SST_i == 0 => R²_i = NaN
        with np.errstate(divide='ignore', invalid='ignore'):
            r2_i = 1.0 - (sse_i / sst_i)
        r2_i[~np.isfinite(r2_i)] = np.nan
        r2_list.append(r2_i)

        # ----- Per-sample PCC across genes -----
        # Center each row
        Xt_c = Xt - Xt.mean(axis=1, keepdims=True)
        Xc_c = Xc - Xc.mean(axis=1, keepdims=True)
        # Row-wise dot product and norms
        num = (Xt_c * Xc_c).sum(axis=1, dtype="float64")
        den = np.sqrt((Xt_c * Xt_c).sum(axis=1, dtype="float64")) * np.sqrt((Xc_c * Xc_c).sum(axis=1, dtype="float64"))
        with np.errstate(divide='ignore', invalid='ignore'):
            pcc_i = num / den
        pcc_i[~np.isfinite(pcc_i)] = np.nan
        pcc_list.append(pcc_i)

    # Finalize aggregates
    overall_mse = float(total_sse / (test_idx_all.size * n_genes))
    r2_all = np.concatenate(r2_list, axis=0)
    pcc_all = np.concatenate(pcc_list, axis=0)

    mean_r2 = float(np.nanmean(r2_all)) if np.any(np.isfinite(r2_all)) else np.nan
    mean_pcc = float(np.nanmean(pcc_all)) if np.any(np.isfinite(pcc_all)) else np.nan

    stats = {
        "n_test_used": int(test_idx_all.size),
        "n_r2_used": int(np.isfinite(r2_all).sum()),
        "n_pcc_used": int(np.isfinite(pcc_all).sum()),
    }
    return overall_mse, mean_r2, mean_pcc, stats


In [7]:
overall_mse_cell, mean_r2_cell, mean_pcc_cell, stats_cell = compute_overall_baseline_metrics(
    adata,
    split_col="cell_split1",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, cell split:", overall_mse_cell)
print("Mean R² across samples, cell split:", mean_r2_cell)
print("Mean PCC across samples, cell split:", mean_pcc_cell)
print("Stats, cell split:", stats_cell)

overall_mse_random, mean_r2_random, mean_pcc_random, stats_random = compute_overall_baseline_metrics(
    adata,
    split_col="random_split1",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, random split:", overall_mse_random)
print("Mean R² across samples, random split:", mean_r2_random)
print("Mean PCC across samples, random split:", mean_pcc_random)
print("Stats, random split:", stats_random)

overall_mse_drug, mean_r2_drug, mean_pcc_drug, stats_drug = compute_overall_baseline_metrics(
    adata,
    split_col="drug_split1",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, drug split:", overall_mse_drug)
print("Mean R² across samples, drug split:", mean_r2_drug)
print("Mean PCC across samples, drug split:", mean_pcc_drug)
print("Stats, drug split:", stats_drug)

Overall MSE, cell split: 1.5500008410097348
Mean R² across samples, cell split: 0.7684370995432493
Mean PCC across samples, cell split: 0.8821235804479689
Stats, cell split: {'n_test_used': 12510, 'n_r2_used': 12510, 'n_pcc_used': 12510}
Overall MSE, random split: 1.373725299147847
Mean R² across samples, random split: 0.7722379753638418
Mean PCC across samples, random split: 0.8875717205271527
Stats, random split: {'n_test_used': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665}
Overall MSE, drug split: 1.3268121510330975
Mean R² across samples, drug split: 0.7813029634725334
Mean PCC across samples, drug split: 0.8919533307182995
Stats, drug split: {'n_test_used': 85539, 'n_r2_used': 85539, 'n_pcc_used': 85539}


In [6]:
overall_mse_cell, mean_r2_cell, mean_pcc_cell, stats_cell = compute_overall_baseline_metrics(
    adata,
    split_col="cell_split2",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, cell split:", overall_mse_cell)
print("Mean R² across samples, cell split:", mean_r2_cell)
print("Mean PCC across samples, cell split:", mean_pcc_cell)
print("Stats, cell split:", stats_cell)

overall_mse_random, mean_r2_random, mean_pcc_random, stats_random = compute_overall_baseline_metrics(
    adata,
    split_col="random_split2",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, random split:", overall_mse_random)
print("Mean R² across samples, random split:", mean_r2_random)
print("Mean PCC across samples, random split:", mean_pcc_random)
print("Stats, random split:", stats_random)

overall_mse_drug, mean_r2_drug, mean_pcc_drug, stats_drug = compute_overall_baseline_metrics(
    adata,
    split_col="drug_split2",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, drug split:", overall_mse_drug)
print("Mean R² across samples, drug split:", mean_r2_drug)
print("Mean PCC across samples, drug split:", mean_pcc_drug)
print("Stats, drug split:", stats_drug)

Overall MSE, cell split: 1.533844730048982
Mean R² across samples, cell split: 0.7528826456268642
Mean PCC across samples, cell split: 0.8769682546362149
Stats, cell split: {'n_test_used': 25742, 'n_r2_used': 25742, 'n_pcc_used': 25742}
Overall MSE, random split: 1.369334645622027
Mean R² across samples, random split: 0.7729329217079944
Mean PCC across samples, random split: 0.8879354725973962
Stats, random split: {'n_test_used': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665}
Overall MSE, drug split: 1.382287127423
Mean R² across samples, drug split: 0.7717263797018648
Mean PCC across samples, drug split: 0.8872501214656836
Stats, drug split: {'n_test_used': 86935, 'n_r2_used': 86935, 'n_pcc_used': 86935}


In [8]:
overall_mse_cell, mean_r2_cell, mean_pcc_cell, stats_cell = compute_overall_baseline_metrics(
    adata,
    split_col="cell_split3",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, cell split:", overall_mse_cell)
print("Mean R² across samples, cell split:", mean_r2_cell)
print("Mean PCC across samples, cell split:", mean_pcc_cell)
print("Stats, cell split:", stats_cell)

overall_mse_random, mean_r2_random, mean_pcc_random, stats_random = compute_overall_baseline_metrics(
    adata,
    split_col="random_split3",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, random split:", overall_mse_random)
print("Mean R² across samples, random split:", mean_r2_random)
print("Mean PCC across samples, random split:", mean_pcc_random)
print("Stats, random split:", stats_random)

overall_mse_drug, mean_r2_drug, mean_pcc_drug, stats_drug = compute_overall_baseline_metrics(
    adata,
    split_col="drug_split3",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, drug split:", overall_mse_drug)
print("Mean R² across samples, drug split:", mean_r2_drug)
print("Mean PCC across samples, drug split:", mean_pcc_drug)
print("Stats, drug split:", stats_drug)

Overall MSE, cell split: 1.406432853330598
Mean R² across samples, cell split: 0.7735214553298629
Mean PCC across samples, cell split: 0.888096534727017
Stats, cell split: {'n_test_used': 156258, 'n_r2_used': 156258, 'n_pcc_used': 156258}
Overall MSE, random split: 1.368764826205406
Mean R² across samples, random split: 0.773385229368735
Mean PCC across samples, random split: 0.8880437394750037
Stats, random split: {'n_test_used': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665}
Overall MSE, drug split: 1.3365611352161693
Mean R² across samples, drug split: 0.7800416708722161
Mean PCC across samples, drug split: 0.8916126295118968
Stats, drug split: {'n_test_used': 78097, 'n_r2_used': 78097, 'n_pcc_used': 78097}


In [9]:
overall_mse_cell, mean_r2_cell, mean_pcc_cell, stats_cell = compute_overall_baseline_metrics(
    adata,
    split_col="cell_split4",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, cell split:", overall_mse_cell)
print("Mean R² across samples, cell split:", mean_r2_cell)
print("Mean PCC across samples, cell split:", mean_pcc_cell)
print("Stats, cell split:", stats_cell)

overall_mse_random, mean_r2_random, mean_pcc_random, stats_random = compute_overall_baseline_metrics(
    adata,
    split_col="random_split4",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, random split:", overall_mse_random)
print("Mean R² across samples, random split:", mean_r2_random)
print("Mean PCC across samples, random split:", mean_pcc_random)
print("Stats, random split:", stats_random)

overall_mse_drug, mean_r2_drug, mean_pcc_drug, stats_drug = compute_overall_baseline_metrics(
    adata,
    split_col="drug_split4",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, drug split:", overall_mse_drug)
print("Mean R² across samples, drug split:", mean_r2_drug)
print("Mean PCC across samples, drug split:", mean_pcc_drug)
print("Stats, drug split:", stats_drug)

Overall MSE, cell split: 1.3075300518861763
Mean R² across samples, cell split: 0.7777085637333332
Mean PCC across samples, cell split: 0.889809839512878
Stats, cell split: {'n_test_used': 137725, 'n_r2_used': 137725, 'n_pcc_used': 137725}
Overall MSE, random split: 1.3779053236910483
Mean R² across samples, random split: 0.7718548299566672
Mean PCC across samples, random split: 0.8872410945759185
Stats, random split: {'n_test_used': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665}
Overall MSE, drug split: 1.3530696646109215
Mean R² across samples, drug split: 0.7785410176764934
Mean PCC across samples, drug split: 0.8903910413059808
Stats, drug split: {'n_test_used': 77889, 'n_r2_used': 77889, 'n_pcc_used': 77889}


In [10]:
overall_mse_cell, mean_r2_cell, mean_pcc_cell, stats_cell = compute_overall_baseline_metrics(
    adata,
    split_col="cell_split5",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, cell split:", overall_mse_cell)
print("Mean R² across samples, cell split:", mean_r2_cell)
print("Mean PCC across samples, cell split:", mean_pcc_cell)
print("Stats, cell split:", stats_cell)

overall_mse_random, mean_r2_random, mean_pcc_random, stats_random = compute_overall_baseline_metrics(
    adata,
    split_col="random_split5",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, random split:", overall_mse_random)
print("Mean R² across samples, random split:", mean_r2_random)
print("Mean PCC across samples, random split:", mean_pcc_random)
print("Stats, random split:", stats_random)

overall_mse_drug, mean_r2_drug, mean_pcc_drug, stats_drug = compute_overall_baseline_metrics(
    adata,
    split_col="drug_split5",   # or "random_split2", "drug_split2"
    batch_size=20000,
)

print("Overall MSE, drug split:", overall_mse_drug)
print("Mean R² across samples, drug split:", mean_r2_drug)
print("Mean PCC across samples, drug split:", mean_pcc_drug)
print("Stats, drug split:", stats_drug)

Overall MSE, cell split: 1.5140573795214372
Mean R² across samples, cell split: 0.7595853550800299
Mean PCC across samples, cell split: 0.8816918143282328
Stats, cell split: {'n_test_used': 86370, 'n_r2_used': 86370, 'n_pcc_used': 86370}
Overall MSE, random split: 1.3767086111207514
Mean R² across samples, random split: 0.7716406538994609
Mean PCC across samples, random split: 0.887267234501825
Stats, random split: {'n_test_used': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665}
Overall MSE, drug split: 1.3483290387721627
Mean R² across samples, drug split: 0.7782641124926858
Mean PCC across samples, drug split: 0.8905607501458518
Stats, drug split: {'n_test_used': 80260, 'n_r2_used': 80260, 'n_pcc_used': 80260}


In [ ]:
'''
use average training gene expression as prediction
'''

In [11]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple, Dict, Any
from scipy.sparse import issparse

def compute_trainmean_baseline_mse(
    adata,
    split_col: str = "random_split1",
    batch_size_test: int = 20000,
    batch_size_train: int = 20000,
    return_per_gene_mse: bool = True,
    return_per_sample_mse: bool = False,
    dtype: str = "float32",
) -> Tuple[Dict[str, Any], Optional[pd.Series], Optional[pd.DataFrame]]:
    """
    Train-mean baseline:
      - Baseline prediction for every TEST row = mean(.X) over TRAIN rows (by `split_col`).

    Returns
    -------
    metrics : dict
        {
          "overall_mse": float,
          "mean_r2": float,          # avg of per-sample R² across genes
          "mean_pcc": float,         # avg of per-sample Pearson r across genes
          "mean_delta_r2": float,    # avg of per-sample R² on Δ (obs - control) vs (pred - control)
          "mean_delta_pcc": float,   # avg of per-sample PCC on Δ
          "counts": {
              "n_test": int,
              "n_r2_used": int,          # samples contributing to R² avg
              "n_pcc_used": int,         # samples contributing to PCC avg
              "n_delta_used": int        # samples contributing to Δ-metrics avg
          }
        }
    per_gene_mse : pd.Series or None
        Mean squared error per gene across TEST rows (if requested).
    per_sample_mse : pd.DataFrame or None
        Per-sample MSE over genes with minimal metadata (if requested).

    Notes
    -----
    - Works with dense or sparse .X; streams TRAIN and TEST to limit memory.
    - Δ-metrics need `paired_control_index` (label or positional); missing/unresolvable controls are skipped for Δ.
    """
    if split_col not in adata.obs.columns:
        raise KeyError(f"'{split_col}' not found in adata.obs")

    split = adata.obs[split_col].astype(str).str.lower()
    train_mask = (split == "train").values
    test_mask  = (split == "test").values

    if not train_mask.any():
        raise ValueError(f"No rows labeled 'train' under '{split_col}'.")
    if not test_mask.any():
        raise ValueError(f"No rows labeled 'test' under '{split_col}'.")

    X = adata.X
    n_genes = adata.n_vars
    n_train = int(train_mask.sum())
    train_idx_all = np.where(train_mask)[0]
    test_idx_all  = np.where(test_mask)[0]

    def _rows_to_array(idx: np.ndarray):
        if issparse(X):
            return X[idx].toarray()
        return np.asarray(X[idx])

    # -------- Compute TRAIN mean vector (streamed) --------
    train_sum64 = np.zeros(n_genes, dtype="float64")
    for s in range(0, train_idx_all.size, batch_size_train):
        e = min(s + batch_size_train, train_idx_all.size)
        block = _rows_to_array(train_idx_all[s:e])
        train_sum64 += block.sum(axis=0, dtype="float64")
    train_mean = (train_sum64 / n_train).astype(dtype, copy=False)

    # -------- Prepare Δ (control) indexing if available --------
    have_paired = "paired_control_index" in adata.obs.columns
    ctrl_abs_idx = None
    if have_paired and test_idx_all.size > 0:
        pci_series = adata.obs.iloc[test_idx_all]["paired_control_index"]
        obs_names = adata.obs_names.astype(str)
        idx_from_labels = pd.Index(obs_names).get_indexer(pci_series.astype(str))
        label_ok = idx_from_labels >= 0

        numeric_try = pd.to_numeric(pci_series[~label_ok], errors="coerce")
        numeric_ok = numeric_try.notna() & (numeric_try.astype(float) >= 0) & (numeric_try.astype(float) < adata.n_obs)

        ctrl_abs_idx = np.full(test_idx_all.size, fill_value=-1, dtype=int)
        if label_ok.any():
            ctrl_abs_idx[label_ok] = idx_from_labels[label_ok]
        if numeric_ok.any():
            pos_num = np.where(~label_ok)[0][numeric_ok.values]
            ctrl_abs_idx[pos_num] = numeric_try.loc[numeric_try.index[numeric_ok]].astype(int).values

        # mark which test rows have a valid control
        valid_ctrl_mask = ctrl_abs_idx >= 0
    else:
        valid_ctrl_mask = np.zeros(test_idx_all.size, dtype=bool)

    # -------- Evaluate on TEST rows (streamed) --------
    total_sse = 0.0
    per_gene_sse = np.zeros(n_genes, dtype="float64") if return_per_gene_mse else None
    per_sample_mse_vals = [] if return_per_sample_mse else None

    r2_list   = []
    pcc_list  = []
    d_r2_list = []  # delta R²
    d_pcc_list= []  # delta PCC

    for s in range(0, test_idx_all.size, batch_size_test):
        e = min(s + batch_size_test, test_idx_all.size)
        ti = test_idx_all[s:e]
        Xt = _rows_to_array(ti).astype(dtype, copy=False)  # observed perturbed (b, G)

        # Prediction = train_mean (broadcast)
        diff = Xt - train_mean
        se = diff * diff

        total_sse += se.sum(dtype="float64")
        if per_gene_sse is not None:
            per_gene_sse += se.sum(axis=0, dtype="float64")
        if per_sample_mse_vals is not None:
            per_sample_mse_vals.append(se.mean(axis=1))  # (b,)

        # ----- Per-sample R² across genes -----
        sse_i = np.square(diff).sum(axis=1, dtype="float64")
        y_mean = Xt.mean(axis=1, dtype="float64", keepdims=True)
        sst_i = np.square(Xt - y_mean).sum(axis=1, dtype="float64")
        with np.errstate(divide='ignore', invalid='ignore'):
            r2_i = 1.0 - (sse_i / sst_i)
        r2_i[~np.isfinite(r2_i)] = np.nan
        r2_list.append(r2_i)

        # ----- Per-sample PCC across genes -----
        Xt_c = Xt - Xt.mean(axis=1, keepdims=True)
        Yhat = np.broadcast_to(train_mean, Xt.shape)
        Yhat_c = Yhat - Yhat.mean(axis=1, keepdims=True)  # rowwise center (same constant per row)
        num = (Xt_c * Yhat_c).sum(axis=1, dtype="float64")
        den = np.sqrt((Xt_c * Xt_c).sum(axis=1, dtype="float64")) * np.sqrt((Yhat_c * Yhat_c).sum(axis=1, dtype="float64"))
        with np.errstate(divide='ignore', invalid='ignore'):
            pcc_i = num / den
        pcc_i[~np.isfinite(pcc_i)] = np.nan
        pcc_list.append(pcc_i)

        # ----- Δ-metrics (only for rows with valid controls) -----
        if valid_ctrl_mask[s:e].any():
            # rows in this batch that have a resolvable control
            rel_mask = valid_ctrl_mask[s:e]
            ti_ctrl = ctrl_abs_idx[s:e][rel_mask]
            Xt_sub = Xt[rel_mask]                              # (k, G)
            Xc_sub = _rows_to_array(ti_ctrl).astype(dtype, copy=False)  # (k, G)

            # Δ observed and Δ predicted
            d_obs = Xt_sub - Xc_sub
            d_pred = np.broadcast_to(train_mean, Xc_sub.shape) - Xc_sub

            # Δ R² per sample
            d_diff = d_obs - d_pred
            d_sse = np.square(d_diff).sum(axis=1, dtype="float64")
            d_y_mean = d_obs.mean(axis=1, dtype="float64", keepdims=True)
            d_sst = np.square(d_obs - d_y_mean).sum(axis=1, dtype="float64")
            with np.errstate(divide='ignore', invalid='ignore'):
                d_r2_i = 1.0 - (d_sse / d_sst)
            d_r2_i[~np.isfinite(d_r2_i)] = np.nan
            d_r2_list.append(d_r2_i)

            # Δ PCC per sample
            d_obs_c  = d_obs  - d_obs.mean(axis=1, keepdims=True)
            d_pred_c = d_pred - d_pred.mean(axis=1, keepdims=True)
            d_num = (d_obs_c * d_pred_c).sum(axis=1, dtype="float64")
            d_den = np.sqrt((d_obs_c * d_obs_c).sum(axis=1, dtype="float64")) * np.sqrt((d_pred_c * d_pred_c).sum(axis=1, dtype="float64"))
            with np.errstate(divide='ignore', invalid='ignore'):
                d_pcc_i = d_num / d_den
            d_pcc_i[~np.isfinite(d_pcc_i)] = np.nan
            d_pcc_list.append(d_pcc_i)

    # -------- Final aggregates --------
    overall_mse = float(total_sse / (test_idx_all.size * n_genes))

    r2_all  = np.concatenate(r2_list,  axis=0) if len(r2_list)  else np.array([], dtype=float)
    pcc_all = np.concatenate(pcc_list, axis=0) if len(pcc_list) else np.array([], dtype=float)

    mean_r2  = float(np.nanmean(r2_all))  if r2_all.size  and np.any(np.isfinite(r2_all))  else np.nan
    mean_pcc = float(np.nanmean(pcc_all)) if pcc_all.size and np.any(np.isfinite(pcc_all)) else np.nan

    if len(d_r2_list):
        d_r2_all  = np.concatenate(d_r2_list,  axis=0)
        d_pcc_all = np.concatenate(d_pcc_list, axis=0)
        mean_delta_r2  = float(np.nanmean(d_r2_all))  if np.any(np.isfinite(d_r2_all))  else np.nan
        mean_delta_pcc = float(np.nanmean(d_pcc_all)) if np.any(np.isfinite(d_pcc_all)) else np.nan
        n_delta_used = int(np.isfinite(d_r2_all).sum())  # same mask scale for both deltas (roughly)
    else:
        mean_delta_r2 = np.nan
        mean_delta_pcc = np.nan
        n_delta_used = 0

    metrics = {
        "overall_mse": overall_mse,
        "mean_r2": mean_r2,
        "mean_pcc": mean_pcc,
        "mean_delta_r2": mean_delta_r2,
        "mean_delta_pcc": mean_delta_pcc,
        "counts": {
            "n_test": int(test_idx_all.size),
            "n_r2_used": int(np.isfinite(r2_all).sum()) if r2_all.size else 0,
            "n_pcc_used": int(np.isfinite(pcc_all).sum()) if pcc_all.size else 0,
            "n_delta_used": n_delta_used,
        }
    }

    per_gene_mse = None
    if return_per_gene_mse:
        per_gene_mse = pd.Series(per_gene_sse / test_idx_all.size, index=adata.var_names, name="per_gene_mse")

    per_sample_mse = None
    if return_per_sample_mse:
        per_sample_mse_arr = np.concatenate(per_sample_mse_vals, axis=0) if len(per_sample_mse_vals) else np.array([], dtype=float)
        meta_cols = [c for c in ["pert_iname","dose","pert_time","cell_id","condition","pert_dose","pert_time_unit"]
                     if c in adata.obs.columns]
        df = adata.obs.iloc[test_idx_all][meta_cols].copy()
        df.insert(0, "obs_name", adata.obs.index.values[test_idx_all])
        df["mse_over_genes"] = per_sample_mse_arr.astype(float)
        per_sample_mse = df

    return metrics, per_gene_mse, per_sample_mse


In [15]:
metrics_cell, per_gene_mse_cell, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="cell_split1",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, cell:", metrics_cell)

metrics_random, per_gene_mse_random, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="random_split1",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, random:", metrics_random)

metrics_drug, per_gene_mse_drug, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="drug_split1",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 3.1071520911833233, 'mean_r2': 0.5142443667107968, 'mean_pcc': 0.720075369377452, 'mean_delta_r2': -1.8733419452380418, 'mean_delta_pcc': 0.31516883230412845, 'counts': {'n_test': 12510, 'n_r2_used': 12510, 'n_pcc_used': 12510, 'n_delta_used': 12510}}
metrics, random: {'overall_mse': 2.271118712000575, 'mean_r2': 0.6174965324589226, 'mean_pcc': 0.78761138457676, 'mean_delta_r2': -1.2986498077298279, 'mean_delta_pcc': 0.3495955954091059, 'counts': {'n_test': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 2.256664409923434, 'mean_r2': 0.6205205300048263, 'mean_pcc': 0.7893910948246851, 'mean_delta_r2': -1.3673911429047434, 'mean_delta_pcc': 0.34673248601787837, 'counts': {'n_test': 85539, 'n_r2_used': 85539, 'n_pcc_used': 85539, 'n_delta_used': 85539}}


In [14]:
metrics_cell, per_gene_mse_cell, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="cell_split2",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, cell:", metrics_cell)

metrics_random, per_gene_mse_random, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="random_split2",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, random:", metrics_random)

metrics_drug, per_gene_mse_drug, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="drug_split2",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.640096130558899, 'mean_r2': 0.554489898322301, 'mean_pcc': 0.747043477525436, 'mean_delta_r2': -1.4534691409728924, 'mean_delta_pcc': 0.3353435821863263, 'counts': {'n_test': 25742, 'n_r2_used': 25742, 'n_pcc_used': 25742, 'n_delta_used': 25742}}
metrics, random: {'overall_mse': 2.2662488929610207, 'mean_r2': 0.61815631379299, 'mean_pcc': 0.7879920518052964, 'mean_delta_r2': -1.2960941878074592, 'mean_delta_pcc': 0.34997197042310857, 'counts': {'n_test': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 2.2764723914162515, 'mean_r2': 0.6168775144763881, 'mean_pcc': 0.7870488019168721, 'mean_delta_r2': -1.2863381299042642, 'mean_delta_pcc': 0.35014972330384037, 'counts': {'n_test': 86935, 'n_r2_used': 86935, 'n_pcc_used': 86935, 'n_delta_used': 86935}}


In [16]:
metrics_cell, per_gene_mse_cell, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="cell_split3",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, cell:", metrics_cell)

metrics_random, per_gene_mse_random, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="random_split3",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, random:", metrics_random)

metrics_drug, per_gene_mse_drug, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="drug_split3",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.666292893867408, 'mean_r2': 0.5703036339656528, 'mean_pcc': 0.7562750853208349, 'mean_delta_r2': -1.5627254308675202, 'mean_delta_pcc': 0.33085047464808404, 'counts': {'n_test': 156258, 'n_r2_used': 156258, 'n_pcc_used': 156258, 'n_delta_used': 156258}}
metrics, random: {'overall_mse': 2.2692503381079594, 'mean_r2': 0.6178692617431818, 'mean_pcc': 0.7878440388217557, 'mean_delta_r2': -1.302400894539322, 'mean_delta_pcc': 0.3497405148811029, 'counts': {'n_test': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 2.249861002449781, 'mean_r2': 0.623117290853182, 'mean_pcc': 0.7910844448968773, 'mean_delta_r2': -1.3321043866228142, 'mean_delta_pcc': 0.3481213432179603, 'counts': {'n_test': 78097, 'n_r2_used': 78097, 'n_pcc_used': 78097, 'n_delta_used': 78097}}


In [17]:
metrics_cell, per_gene_mse_cell, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="cell_split4",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, cell:", metrics_cell)

metrics_random, per_gene_mse_random, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="random_split4",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, random:", metrics_random)

metrics_drug, per_gene_mse_drug, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="drug_split4",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.2639684444438166, 'mean_r2': 0.605187275803752, 'mean_pcc': 0.7804628076294486, 'mean_delta_r2': -1.3548697994594694, 'mean_delta_pcc': 0.3412161819655501, 'counts': {'n_test': 137725, 'n_r2_used': 137725, 'n_pcc_used': 137725, 'n_delta_used': 137725}}
metrics, random: {'overall_mse': 2.274198653607464, 'mean_r2': 0.6170140480300541, 'mean_pcc': 0.7872495731705431, 'mean_delta_r2': -1.2918574260730247, 'mean_delta_pcc': 0.349890454644112, 'counts': {'n_test': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 2.282735166498102, 'mean_r2': 0.6188786863163602, 'mean_pcc': 0.7882188297673445, 'mean_delta_r2': -1.3589893379779532, 'mean_delta_pcc': 0.34560678539252393, 'counts': {'n_test': 77889, 'n_r2_used': 77889, 'n_pcc_used': 77889, 'n_delta_used': 77889}}


In [18]:
metrics_cell, per_gene_mse_cell, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="cell_split5",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, cell:", metrics_cell)

metrics_random, per_gene_mse_random, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="random_split5",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, random:", metrics_random)

metrics_drug, per_gene_mse_drug, _ = compute_trainmean_baseline_mse(
    adata,
    split_col="drug_split5",           # or "random_split2", "drug_split2"
    return_per_gene_mse=True,
    return_per_sample_mse=False,
)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.2976518826715093, 'mean_r2': 0.627578538387454, 'mean_pcc': 0.7946216041811333, 'mean_delta_r2': -1.040495459634612, 'mean_delta_pcc': 0.3680082977043686, 'counts': {'n_test': 86370, 'n_r2_used': 86370, 'n_pcc_used': 86370, 'n_delta_used': 86370}}
metrics, random: {'overall_mse': 2.2690405558253297, 'mean_r2': 0.6174857279736637, 'mean_pcc': 0.7875925199608058, 'mean_delta_r2': -1.2901469989861059, 'mean_delta_pcc': 0.3501944211968816, 'counts': {'n_test': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 2.258418342591944, 'mean_r2': 0.6222856538292216, 'mean_pcc': 0.7905133015417812, 'mean_delta_r2': -1.318739258443769, 'mean_delta_pcc': 0.3489203437121519, 'counts': {'n_test': 80260, 'n_r2_used': 80260, 'n_pcc_used': 80260, 'n_delta_used': 80260}}


In [19]:
import numpy as np
import pandas as pd
from typing import Optional, Literal, Dict, Any
from scipy.sparse import issparse

def meta_metrics_by_split(
    adata,
    split_col: str,
    mode: Optional[Literal["random","drug","cell"]] = None,
    fallback: Literal["global","drop","error"] = "global",
    batch_size_test: int = 20000,
    dtype: str = "float32",
) -> Dict[str, Any]:
    """
    Train-mean (by TRAIN only) baseline evaluation for a given split scheme.
    Returns averaged metrics across TEST samples:

      overall_mse      : float  (mean over all genes & all kept TEST samples)
      mean_r2          : float  (avg of per-sample R^2 across genes)
      mean_pcc         : float  (avg of per-sample Pearson r across genes)
      mean_delta_r2    : float  (avg per-sample R^2 on Δ=obs-control vs Δ̂=pred-control)
      mean_delta_pcc   : float  (avg per-sample Pearson r on Δ)

    Modes (baseline from TRAIN):
      - "random": baseline = mean .X for (cell_id, pert_iname)
      - "drug"  : baseline = mean .X for cell_id
      - "cell"  : baseline = mean .X for pert_iname

    Fallback when a TEST key has no TRAIN baseline:
      - "global": use global TRAIN mean
      - "drop"  : drop the sample from all metrics
      - "error" : raise

    Notes:
      - Δ-metrics require `obs['paired_control_index']` resolvable to control rows.
        Samples without a valid control are skipped for Δ-metrics only.
    """
    # --- checks
    if split_col not in adata.obs.columns:
        raise KeyError(f"'{split_col}' not found in adata.obs")
    for col in ["cell_id", "pert_iname"]:
        if col not in adata.obs.columns:
            raise KeyError(f"'{col}' not found in adata.obs")

    # --- infer mode if needed
    if mode is None:
        sc = split_col.lower()
        if "random_split" in sc:
            mode = "random"
        elif "drug_split" in sc:
            mode = "drug"
        elif "cell_split" in sc:
            mode = "cell"
        else:
            raise ValueError("Unable to infer mode from split_col. Pass mode in {'random','drug','cell'}.")

    split_vals = adata.obs[split_col].astype(str).str.lower().values
    train_mask = (split_vals == "train")
    test_mask  = (split_vals == "test")
    if not train_mask.any():
        raise ValueError(f"No TRAIN rows under '{split_col}'.")
    if not test_mask.any():
        raise ValueError(f"No TEST rows under '{split_col}'.")

    X = adata.X
    n_genes = adata.n_vars
    train_abs_idx = np.where(train_mask)[0]
    test_abs_idx  = np.where(test_mask)[0]

    def _rows_to_array(row_idx: np.ndarray):
        if issparse(X):
            return X[row_idx].toarray()
        return np.asarray(X[row_idx])

    # --- global TRAIN mean (also used for fallback)
    gsum = np.zeros(n_genes, dtype="float64")
    for s in range(0, train_abs_idx.size, 50000):
        e = min(s + 50000, train_abs_idx.size)
        gsum += _rows_to_array(train_abs_idx[s:e]).sum(axis=0, dtype="float64")
    global_mean = (gsum / train_abs_idx.size).astype(dtype, copy=False)

    # --- build TRAIN group means
    train_keys_df = adata.obs.iloc[train_abs_idx][["cell_id","pert_iname"]].copy()
    train_keys_df["obs_idx"] = train_abs_idx
    group_map = {}

    if mode == "random":
        for key, g in train_keys_df.groupby(["cell_id","pert_iname"], observed=True):
            abs_idx = g["obs_idx"].to_numpy()
            ssum = np.zeros(n_genes, dtype="float64")
            for s in range(0, abs_idx.size, 50000):
                e = min(s + 50000, abs_idx.size)
                ssum += _rows_to_array(abs_idx[s:e]).sum(axis=0, dtype="float64")
            group_map[key] = (ssum / abs_idx.size).astype(dtype, copy=False)

    elif mode == "drug":
        for cell, g in train_keys_df.groupby("cell_id", observed=True):
            abs_idx = g["obs_idx"].to_numpy()
            ssum = np.zeros(n_genes, dtype="float64")
            for s in range(0, abs_idx.size, 50000):
                e = min(s + 50000, abs_idx.size)
                ssum += _rows_to_array(abs_idx[s:e]).sum(axis=0, dtype="float64")
            group_map[str(cell)] = (ssum / abs_idx.size).astype(dtype, copy=False)

    elif mode == "cell":
        for drug, g in train_keys_df.groupby("pert_iname", observed=True):
            abs_idx = g["obs_idx"].to_numpy()
            ssum = np.zeros(n_genes, dtype="float64")
            for s in range(0, abs_idx.size, 50000):
                e = min(s + 50000, abs_idx.size)
                ssum += _rows_to_array(abs_idx[s:e]).sum(axis=0, dtype="float64")
            group_map[str(drug)] = (ssum / abs_idx.size).astype(dtype, copy=False)
    else:
        raise ValueError("mode must be one of {'random','drug','cell'}.")

    # --- Resolve paired controls for Δ-metrics (if available)
    have_paired = "paired_control_index" in adata.obs.columns
    ctrl_abs_idx = None
    valid_ctrl_mask = np.zeros(test_abs_idx.size, dtype=bool)
    if have_paired and test_abs_idx.size > 0:
        pci_series = adata.obs.iloc[test_abs_idx]["paired_control_index"]
        obs_names = adata.obs_names.astype(str)
        idx_from_labels = pd.Index(obs_names).get_indexer(pci_series.astype(str))
        label_ok = idx_from_labels >= 0

        numeric_try = pd.to_numeric(pci_series[~label_ok], errors="coerce")
        numeric_ok = numeric_try.notna() & (numeric_try.astype(float) >= 0) & (numeric_try.astype(float) < adata.n_obs)

        ctrl_abs_idx = np.full(test_abs_idx.size, fill_value=-1, dtype=int)
        if label_ok.any():
            ctrl_abs_idx[label_ok] = idx_from_labels[label_ok]
        if numeric_ok.any():
            pos_num = np.where(~label_ok)[0][numeric_ok.values]
            ctrl_abs_idx[pos_num] = numeric_try.loc[numeric_try.index[numeric_ok]].astype(int).values

        valid_ctrl_mask = ctrl_abs_idx >= 0

    # --- stream TEST rows to accumulate metrics
    test_obs = adata.obs.iloc[test_abs_idx]
    cell_ids = test_obs["cell_id"].astype(str).values
    drugs    = test_obs["pert_iname"].astype(str).values

    total_sse = 0.0
    total_count = 0

    r2_list, pcc_list = [], []
    d_r2_list, d_pcc_list = [], []

    for start in range(0, test_abs_idx.size, batch_size_test):
        end = min(start + batch_size_test, test_abs_idx.size)
        ti = test_abs_idx[start:end]
        Xt = _rows_to_array(ti).astype(dtype, copy=False)  # (b, G)

        # Build baselines for this batch & keep mask
        baselines = np.empty_like(Xt)
        keep_mask = np.ones(Xt.shape[0], dtype=bool)

        for j in range(Xt.shape[0]):
            if mode == "random":
                key = (cell_ids[start + j], drugs[start + j])
                baseline = group_map.get(key, None)
            elif mode == "drug":
                key = cell_ids[start + j]
                baseline = group_map.get(str(key), None)
            else:  # "cell"
                key = drugs[start + j]
                baseline = group_map.get(str(key), None)

            if baseline is None:
                if fallback == "global":
                    baseline = global_mean
                elif fallback == "drop":
                    keep_mask[j] = False
                    continue
                else:  # "error"
                    raise KeyError(f"No TRAIN baseline for key {key} (mode={mode}).")

            baselines[j] = baseline

        if not keep_mask.any():
            continue

        Xt_k = Xt[keep_mask]
        Yhat = baselines[keep_mask]

        # ------- MSE aggregate (over all genes & kept samples)
        diff = Xt_k - Yhat
        se = diff * diff
        total_sse += se.sum(dtype="float64")
        total_count += Xt_k.shape[0] * n_genes

        # ------- per-sample R^2 across genes
        sse_i = np.square(Xt_k - Yhat).sum(axis=1, dtype="float64")
        y_mean = Xt_k.mean(axis=1, dtype="float64", keepdims=True)
        sst_i = np.square(Xt_k - y_mean).sum(axis=1, dtype="float64")
        with np.errstate(divide='ignore', invalid='ignore'):
            r2_i = 1.0 - (sse_i / sst_i)
        r2_i[~np.isfinite(r2_i)] = np.nan
        r2_list.append(r2_i)

        # ------- per-sample PCC across genes
        Xt_c   = Xt_k - Xt_k.mean(axis=1, keepdims=True)
        Yhat_c = Yhat - Yhat.mean(axis=1, keepdims=True)
        num = (Xt_c * Yhat_c).sum(axis=1, dtype="float64")
        den = np.sqrt((Xt_c * Xt_c).sum(axis=1, dtype="float64")) * np.sqrt((Yhat_c * Yhat_c).sum(axis=1, dtype="float64"))
        with np.errstate(divide='ignore', invalid='ignore'):
            pcc_i = num / den
        pcc_i[~np.isfinite(pcc_i)] = np.nan
        pcc_list.append(pcc_i)

        # ------- Δ-metrics for rows with resolvable controls
        if ctrl_abs_idx is not None:
            rel_mask = valid_ctrl_mask[start:end] & keep_mask
            if rel_mask.any():
                ci_batch = ctrl_abs_idx[start:end][rel_mask]
                Xc = _rows_to_array(ci_batch).astype(dtype, copy=False)

                d_obs  = Xt[rel_mask]  - Xc
                d_pred = Yhat[rel_mask] - Xc

                # Δ R^2
                d_diff = d_obs - d_pred
                d_sse  = np.square(d_diff).sum(axis=1, dtype="float64")
                d_mean = d_obs.mean(axis=1, dtype="float64", keepdims=True)
                d_sst  = np.square(d_obs - d_mean).sum(axis=1, dtype="float64")
                with np.errstate(divide='ignore', invalid='ignore'):
                    d_r2_i = 1.0 - (d_sse / d_sst)
                d_r2_i[~np.isfinite(d_r2_i)] = np.nan
                d_r2_list.append(d_r2_i)

                # Δ PCC
                d_obs_c  = d_obs  - d_obs.mean(axis=1, keepdims=True)
                d_pred_c = d_pred - d_pred.mean(axis=1, keepdims=True)
                d_num = (d_obs_c * d_pred_c).sum(axis=1, dtype="float64")
                d_den = np.sqrt((d_obs_c * d_obs_c).sum(axis=1, dtype="float64")) * np.sqrt((d_pred_c * d_pred_c).sum(axis=1, dtype="float64"))
                with np.errstate(divide='ignore', invalid='ignore'):
                    d_pcc_i = d_num / d_den
                d_pcc_i[~np.isfinite(d_pcc_i)] = np.nan
                d_pcc_list.append(d_pcc_i)

    if total_count == 0:
        raise ValueError("No TEST rows contributed to the metrics (check fallback='drop' coverage).")

    # --- finalize averages across samples
    overall_mse = float(total_sse / total_count)

    r2_all  = np.concatenate(r2_list,  axis=0) if len(r2_list)  else np.array([], dtype=float)
    pcc_all = np.concatenate(pcc_list, axis=0) if len(pcc_list) else np.array([], dtype=float)

    mean_r2  = float(np.nanmean(r2_all))  if r2_all.size  and np.any(np.isfinite(r2_all))  else np.nan
    mean_pcc = float(np.nanmean(pcc_all)) if pcc_all.size and np.any(np.isfinite(pcc_all)) else np.nan

    if len(d_r2_list):
        d_r2_all  = np.concatenate(d_r2_list,  axis=0)
        d_pcc_all = np.concatenate(d_pcc_list, axis=0)
        mean_delta_r2  = float(np.nanmean(d_r2_all))  if np.any(np.isfinite(d_r2_all))  else np.nan
        mean_delta_pcc = float(np.nanmean(d_pcc_all)) if np.any(np.isfinite(d_pcc_all)) else np.nan
        n_delta_used   = int(np.isfinite(d_r2_all).sum())
    else:
        mean_delta_r2 = np.nan
        mean_delta_pcc = np.nan
        n_delta_used = 0

    return {
        "overall_mse": overall_mse,
        "mean_r2": mean_r2,
        "mean_pcc": mean_pcc,
        "mean_delta_r2": mean_delta_r2,
        "mean_delta_pcc": mean_delta_pcc,
        "counts": {
            "n_test": int(test_abs_idx.size),
            "n_used_for_mse": int(total_count // n_genes),
            "n_r2_used": int(np.isfinite(r2_all).sum()) if r2_all.size else 0,
            "n_pcc_used": int(np.isfinite(pcc_all).sum()) if pcc_all.size else 0,
            "n_delta_used": n_delta_used,
        }
    }


In [21]:
metrics_random = meta_metrics_by_split(adata, split_col="random_split1", mode="random", fallback="global")
metrics_drug   = meta_metrics_by_split(adata, split_col="drug_split1",   mode="drug",   fallback="global")
metrics_cell   = meta_metrics_by_split(adata, split_col="cell_split1",   mode="cell",   fallback="global")

print("metrics, cell:", metrics_cell)
print("metrics, random:", metrics_random)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.820120671660881, 'mean_r2': 0.5604773006006165, 'mean_pcc': 0.7504779884886067, 'mean_delta_r2': -1.6266318079819952, 'mean_delta_pcc': 0.34363794965127115, 'counts': {'n_test': 12510, 'n_used_for_mse': 12510, 'n_r2_used': 12510, 'n_pcc_used': 12510, 'n_delta_used': 12510}}
metrics, random: {'overall_mse': 1.3858426136899893, 'mean_r2': 0.7751304271874506, 'mean_pcc': 0.883407649140672, 'mean_delta_r2': -0.22478862099850147, 'mean_delta_pcc': 0.5002551175646217, 'counts': {'n_test': 83665, 'n_used_for_mse': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 1.3833365039437906, 'mean_r2': 0.7737687056892081, 'mean_pcc': 0.8796999254972933, 'mean_delta_r2': -0.28248805875603267, 'mean_delta_pcc': 0.46319110309559747, 'counts': {'n_test': 85539, 'n_used_for_mse': 85539, 'n_r2_used': 85539, 'n_pcc_used': 85539, 'n_delta_used': 85539}}


In [20]:
metrics_random = meta_metrics_by_split(adata, split_col="random_split2", mode="random", fallback="global")
metrics_drug   = meta_metrics_by_split(adata, split_col="drug_split2",   mode="drug",   fallback="global")
metrics_cell   = meta_metrics_by_split(adata, split_col="cell_split2",   mode="cell",   fallback="global")

print("metrics, cell:", metrics_cell)
print("metrics, random:", metrics_random)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.417816702683611, 'mean_r2': 0.5936733783666553, 'mean_pcc': 0.7722593100794388, 'mean_delta_r2': -1.2253535874010033, 'mean_delta_pcc': 0.3606820205983919, 'counts': {'n_test': 25742, 'n_used_for_mse': 25742, 'n_r2_used': 25742, 'n_pcc_used': 25742, 'n_delta_used': 25742}}
metrics, random: {'overall_mse': 1.377721947865692, 'mean_r2': 0.7764999141836794, 'mean_pcc': 0.8839571094327591, 'mean_delta_r2': -0.22092229537225314, 'mean_delta_pcc': 0.5004701020528671, 'counts': {'n_test': 83665, 'n_used_for_mse': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 1.4360022351826527, 'mean_r2': 0.7644700910079841, 'mean_pcc': 0.8743534847338073, 'mean_delta_r2': -0.27815985545771527, 'mean_delta_pcc': 0.459624995236836, 'counts': {'n_test': 86935, 'n_used_for_mse': 86935, 'n_r2_used': 86935, 'n_pcc_used': 86935, 'n_delta_used': 86935}}


In [22]:
metrics_random = meta_metrics_by_split(adata, split_col="random_split3", mode="random", fallback="global")
metrics_drug   = meta_metrics_by_split(adata, split_col="drug_split3",   mode="drug",   fallback="global")
metrics_cell   = meta_metrics_by_split(adata, split_col="cell_split3",   mode="cell",   fallback="global")

print("metrics, cell:", metrics_cell)
print("metrics, random:", metrics_random)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.5252289630991704, 'mean_r2': 0.5960987732260484, 'mean_pcc': 0.7737599306096002, 'mean_delta_r2': -1.4231792135664796, 'mean_delta_pcc': 0.35428324460277416, 'counts': {'n_test': 156258, 'n_used_for_mse': 156258, 'n_r2_used': 156258, 'n_pcc_used': 156258, 'n_delta_used': 156258}}
metrics, random: {'overall_mse': 1.3825595169634937, 'mean_r2': 0.7759165993445098, 'mean_pcc': 0.8838280744946219, 'mean_delta_r2': -0.225164427319729, 'mean_delta_pcc': 0.5008806105665439, 'counts': {'n_test': 83665, 'n_used_for_mse': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 1.3943918013438559, 'mean_r2': 0.7727214159328776, 'mean_pcc': 0.8791570498891087, 'mean_delta_r2': -0.28495440950833956, 'mean_delta_pcc': 0.4624035438098797, 'counts': {'n_test': 78097, 'n_used_for_mse': 78097, 'n_r2_used': 78097, 'n_pcc_used': 78097, 'n_delta_used': 78097}}


In [23]:
metrics_random = meta_metrics_by_split(adata, split_col="random_split4", mode="random", fallback="global")
metrics_drug   = meta_metrics_by_split(adata, split_col="drug_split4",   mode="drug",   fallback="global")
metrics_cell   = meta_metrics_by_split(adata, split_col="cell_split4",   mode="cell",   fallback="global")

print("metrics, cell:", metrics_cell)
print("metrics, random:", metrics_random)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.22411134087995, 'mean_r2': 0.6133158543437243, 'mean_pcc': 0.786608365010439, 'mean_delta_r2': -1.3430366504971638, 'mean_delta_pcc': 0.358214642690703, 'counts': {'n_test': 137725, 'n_used_for_mse': 137725, 'n_r2_used': 137725, 'n_pcc_used': 137725, 'n_delta_used': 137725}}
metrics, random: {'overall_mse': 1.392279698857086, 'mean_r2': 0.7743883662179859, 'mean_pcc': 0.8829403376237218, 'mean_delta_r2': -0.22448744672888277, 'mean_delta_pcc': 0.500019420337582, 'counts': {'n_test': 83665, 'n_used_for_mse': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 1.436621193428784, 'mean_r2': 0.7664092628665261, 'mean_pcc': 0.8754184700062627, 'mean_delta_r2': -0.3145761170376539, 'mean_delta_pcc': 0.4567586538108879, 'counts': {'n_test': 77889, 'n_used_for_mse': 77889, 'n_r2_used': 77889, 'n_pcc_used': 77889, 'n_delta_used': 77889}}


In [24]:
metrics_random = meta_metrics_by_split(adata, split_col="random_split5", mode="random", fallback="global")
metrics_drug   = meta_metrics_by_split(adata, split_col="drug_split5",   mode="drug",   fallback="global")
metrics_cell   = meta_metrics_by_split(adata, split_col="cell_split5",   mode="cell",   fallback="global")

print("metrics, cell:", metrics_cell)
print("metrics, random:", metrics_random)
print("metrics, drug:", metrics_drug)

metrics, cell: {'overall_mse': 2.271256723180033, 'mean_r2': 0.6330473094565964, 'mean_pcc': 0.7980889707943484, 'mean_delta_r2': -1.0096162041462318, 'mean_delta_pcc': 0.3810043576408127, 'counts': {'n_test': 86370, 'n_used_for_mse': 86370, 'n_r2_used': 86370, 'n_pcc_used': 86370, 'n_delta_used': 86370}}
metrics, random: {'overall_mse': 1.3809095484628922, 'mean_r2': 0.7758880145840747, 'mean_pcc': 0.8836553479192835, 'mean_delta_r2': -0.2172881752914708, 'mean_delta_pcc': 0.5009930635608726, 'counts': {'n_test': 83665, 'n_used_for_mse': 83665, 'n_r2_used': 83665, 'n_pcc_used': 83665, 'n_delta_used': 83665}}
metrics, drug: {'overall_mse': 1.3924597142292787, 'mean_r2': 0.7734132578810056, 'mean_pcc': 0.8795883589468797, 'mean_delta_r2': -0.2673128188810483, 'mean_delta_pcc': 0.46490212134343956, 'counts': {'n_test': 80260, 'n_used_for_mse': 80260, 'n_r2_used': 80260, 'n_pcc_used': 80260, 'n_delta_used': 80260}}
